# exp12) association rule mining

ML Lab experiment notebook. Run the code cell below in Google Colab.

In [ ]:
# Experiment 12: Association Rule Mining using Apriori
# A small self-contained implementation; no mlxtend dependency is required.

from itertools import combinations

transactions = [
    {"bread", "milk"},
    {"bread", "diaper", "beer", "eggs"},
    {"milk", "diaper", "beer", "cola"},
    {"bread", "milk", "diaper", "beer"},
    {"bread", "milk", "diaper", "cola"},
    {"bread", "milk", "eggs"},
    {"milk", "diaper", "beer"},
    {"bread", "beer"},
]

min_support = 0.25
min_confidence = 0.60
transaction_count = len(transactions)

def support(itemset):
    return sum(itemset.issubset(transaction) for transaction in transactions) / transaction_count

# Generate frequent itemsets level by level (Apriori principle)
items = sorted(set().union(*transactions))
frequent_itemsets = {}
current_level = [{item} for item in items]
k = 1

while current_level:
    frequent_level = [
        itemset for itemset in current_level
        if support(itemset) >= min_support
    ]
    if not frequent_level:
        break

    for itemset in frequent_level:
        frequent_itemsets[frozenset(itemset)] = support(itemset)

    k += 1
    candidates = set()
    previous = [set(itemset) for itemset in frequent_level]
    for left, right in combinations(previous, 2):
        candidate = left | right
        if len(candidate) == k and all(
            frozenset(subset) in frequent_itemsets
            for subset in combinations(candidate, k - 1)
        ):
            candidates.add(frozenset(candidate))
    current_level = [set(candidate) for candidate in candidates]

print("Frequent itemsets:")
for itemset, item_support in sorted(
    frequent_itemsets.items(), key=lambda pair: (len(pair[0]), sorted(pair[0]))
):
    print(f"{sorted(itemset)}: support={item_support:.2f}")

print("\nAssociation rules:")
for itemset, item_support in frequent_itemsets.items():
    if len(itemset) < 2:
        continue
    item_list = list(itemset)
    for size in range(1, len(item_list)):
        for antecedent_tuple in combinations(item_list, size):
            antecedent = frozenset(antecedent_tuple)
            consequent = itemset - antecedent
            confidence = item_support / frequent_itemsets[antecedent]
            if confidence >= min_confidence:
                consequent_support = frequent_itemsets.get(consequent, support(set(consequent)))
                lift = confidence / consequent_support if consequent_support else 0
                print(f"{sorted(antecedent)} -> {sorted(consequent)} "
                      f"(support={item_support:.2f}, confidence={confidence:.2f}, lift={lift:.2f})")
